In [6]:
import pandas as pd 
from sklearn.cluster import KMeans
import numpy as np
data=pd.read_csv('train_data.csv')
df=data.copy()
df.describe()

,id,prix,nb_chambres,nb_sdb,m2_interieur,m2_jardin,m2_etage,m2_soussol,nb_etages,vue_mer,vue_note,etat_note,design_note,annee_construction,annee_renovation,m2_interieur_15voisins,m2_jardin_15voisins,zipcode,lat,long
count,1.714700e+04,1.714700e+04,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000,17147.000000
mean,4.605475e+09,5.431939e+05,3.371669,2.123010,193.732114,1411.998121,166.652667,27.079448,1.497638,0.008048,0.235376,3.409343,7.668630,1971.154721,86.124453,184.852674,1187.418834,98077.654808,47.560131,-122.213735
std,2.879726e+09,3.716087e+05,0.932029,0.772906,85.587837,3879.062125,77.312579,41.221029,0.542015,0.089352,0.767578,0.649535,1.181903,29.378196,405.556968,64.023277,2504.936376,53.455894,0.138478,0.140614
min,1.000102e+06,7.500000e+04,0.000000,0.000000,26.941657,48.309179,26.941657,0.000000,1.000000,0.000000,0.000000,1.000000,1.000000,1900.000000,0.000000,37.068004,61.222594,98001.000000,47.155900,-122.519000
25%,2.126059e+09,3.230000e+05,3.000000,1.750000,132.850242,468.227425,111.482720,0.000000,1.000000,0.000000,0.000000,3.000000,7.000000,1951.500000,0.000000,138.424378,473.801561,98033.000000,47.472300,-122.328000
50%,3.905120e+09,4.500000e+05,3.000000,2.250000,178.372352,706.150130,145.856559,0.000000,1.500000,0.000000,0.000000,3.000000,7.000000,1975.000000,0.000000,170.940171,707.822371,98065.000000,47.572000,-122.230000
75%,7.339452e+09,6.460000e+05,4.000000,2.500000,236.900780,992.985879,206.243032,52.025269,2.000000,0.000000,0.000000,4.000000,8.000000,1997.000000,0.000000,219.249350,936.780007,98117.000000,47.678100,-122.124500
max,9.900000e+09,7.062500e+06,33.000000,7.750000,932.738759,153414.994426,823.114084,447.788926,3.500000,1.000000,4.000000,5.000000,13.000000,2015.000000,2015.000000,576.923077,80936.454849,98199.000000,47.777600,-121.315000


In [12]:
# Filter out unrealistic data: properties with large interior area but low price
mask = ~((df['m2_interieur'] > 600) & (df['prix'] < 1000000))
df = df[mask]

mask = ~((df['m2_jardin'] > 35000))
df = df[mask]


#Handle Dates
df["date"] = pd.to_datetime(df["date"])  # Convert to datetime
df["year_sold"] = df["date"].dt.year
df["month_sold"] = df["date"].dt.month
df["day_of_week_sold"] = df["date"].dt.dayofweek

df["mean_price_zipcode"] = df.groupby("zipcode")["prix"].transform("mean")

df["house_age"] = df["year_sold"] - df["annee_construction"]
df["years_since_renovation"] = np.where(df["annee_renovation"] == 0, 0, df["year_sold"] - df["annee_renovation"])

df["total_m2"] = df["m2_interieur"] + df["m2_jardin"] + df["m2_soussol"]


df["m2_ratio"] = df["m2_interieur"] / (df["m2_jardin"] + 1e-5)  
df["bathroom_per_bedroom"] = df["nb_sdb"] / (df["nb_chambres"] + 1)
df["floor_utilization"] = df["m2_etage"] / (df["m2_interieur"] + 1e-5)

# Bathroom density
df['bathroom_density'] = df['nb_sdb'] / df['m2_interieur']


df['market_trend'] = df.groupby('year_sold')['prix'].transform('mean')


# Quality-adjusted space
df['quality_adjusted_space'] = df['m2_interieur'] * (df['etat_note'] / 3)

# Neighborhood garden ratio
df['neighborhood_garden_ratio'] = df['m2_jardin'] / df.groupby('zipcode')['m2_jardin'].transform('mean')

# Price per square meter in neighborhood
df['neighborhood_price_per_m2'] = df['mean_price_zipcode'] / df.groupby('zipcode')['m2_interieur'].transform('mean')


# Zipcode average interior square meters
df['zipcode_avg_m2'] = df.groupby('zipcode')['m2_interieur'].transform('mean')

# How much larger/smaller than neighborhood average
df['relative_size'] = df['m2_interieur'] / df['zipcode_avg_m2']

# Square and interaction terms for key numerical features
df['m2_interior_squared'] = df['m2_interieur'] ** 2
df['bedrooms_bathrooms_interaction'] = df['nb_chambres'] * df['nb_sdb']
# df['total_m2_squared'] = df['total_m2'] ** 2

# Create a dummy variable for presence of basement
df['has_basement'] = (df['m2_soussol'] > 0).astype(int)

# Calculate basement percentage of total area
df['basement_pct'] = df['m2_soussol'] / (df['m2_interieur'] + 1e-5)

# # Interaction between basement and house age
df['basement_age_interaction'] = df['has_basement'] * df['house_age']



coords = df[['lat', 'long']]

# Apply k-means clustering
kmeans = KMeans(n_clusters=4, random_state=0).fit(coords)
df['geo_cluster'] = kmeans.labels_


# Add distance to city center (downtown Seattle)
city_center_lat = 47.624161
city_center_long = -122.225083

city_center_lat2 = 47.569200
city_center_long2 = -122.190665


# Calculate Euclidean distance to city center (in coordinate units)
df['distance_to_center'] = np.sqrt((df['lat'] - city_center_lat)**2 + 
                                 (df['long'] - city_center_long)**2)

df['distance_to_center2'] = np.sqrt((df['lat'] - city_center_lat2)**2 + 
                                 (df['long'] - city_center_long2)**2)


# Calculate approximate distance in kilometers (Haversine formula)
def haversine_distance(lat1, lon1, lat2, lon2):
    # Convert decimal degrees to radians
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    
    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    r = 6371  # Radius of Earth in kilometers
    return c * r

df['distance_to_center_km'] = haversine_distance(
    df['lat'], df['long'], city_center_lat, city_center_long
)

df['distance_to_center_km2'] = haversine_distance(
    df['lat'], df['long'], city_center_lat2, city_center_long2
)





columns_list = df.columns.tolist()
print(columns_list)

# X = df.drop(columns=["prix", "id","has_basement","date","zipcode","lat",'long', 'annee_construction', 'annee_renovation', 'year_sold', 'month_sold', 'day_of_week_sold','total_m2', "m2_etage"])
X = df.drop(columns=["prix", "id","has_basement","date","zipcode","lat",'long', 'annee_construction', 'month_sold', 'day_of_week_sold','total_m2'])




num_columns = X.shape[1]
print(f"Number of columns in X: {num_columns}")

# X.describe()

# Show full dataframe info including all columns
print("Full DataFrame Information:")
X.describe()


['id', 'date', 'prix', 'nb_chambres', 'nb_sdb', 'm2_interieur', 'm2_jardin', 'm2_etage', 'm2_soussol', 'nb_etages', 'vue_mer', 'vue_note', 'etat_note', 'design_note', 'annee_construction', 'annee_renovation', 'm2_interieur_15voisins', 'm2_jardin_15voisins', 'zipcode', 'lat', 'long', 'year_sold', 'month_sold', 'day_of_week_sold', 'mean_price_zipcode', 'house_age', 'years_since_renovation', 'total_m2', 'm2_ratio', 'bathroom_per_bedroom', 'floor_utilization', 'bathroom_density', 'market_trend', 'quality_adjusted_space', 'neighborhood_garden_ratio', 'neighborhood_price_per_m2', 'zipcode_avg_m2', 'relative_size', 'm2_interior_squared', 'bedrooms_bathrooms_interaction', 'has_basement', 'basement_pct', 'basement_age_interaction', 'geo_cluster', 'distance_to_center', 'distance_to_center2', 'distance_to_center_km', 'distance_to_center_km2']
Number of columns in X: 37
Full DataFrame Information:


,nb_chambres,nb_sdb,m2_interieur,m2_jardin,m2_etage,m2_soussol,nb_etages,vue_mer,vue_note,etat_note,...,relative_size,m2_interior_squared,bedrooms_bathrooms_interaction,basement_pct,basement_age_interaction,geo_cluster,distance_to_center,distance_to_center2,distance_to_center_km,distance_to_center_km2
count,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,...,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000,17108.000000
mean,3.371873,2.121771,193.537947,1293.666851,166.463778,27.074169,1.497282,0.008008,0.233400,3.409399,...,1.000000,44734.456431,7.524448,0.124255,19.309271,1.618599,0.183712,0.181451,17.375353,16.994023
std,0.931642,0.771248,85.310873,2675.360029,77.114536,41.181445,0.542065,0.089131,0.764526,0.649319,...,0.389911,45342.899872,4.195551,0.171117,29.938801,1.225696,0.095762,0.080298,9.780316,7.878444
min,0.000000,0.000000,26.941657,48.309179,26.941657,0.000000,1.000000,0.000000,0.000000,1.000000,...,0.128498,725.852902,0.000000,0.000000,-1.000000,0.000000,0.002420,0.001335,0.258276,0.100156
25%,3.000000,1.750000,132.850242,467.460981,111.482720,0.000000,1.000000,0.000000,0.000000,3.000000,...,0.721046,17649.186679,4.500000,0.000000,0.000000,0.000000,0.115141,0.125094,10.659844,11.741627
50%,3.000000,2.250000,178.372352,705.174656,145.856559,0.000000,1.500000,0.000000,0.000000,3.000000,...,0.937310,31816.696060,7.000000,0.000000,0.000000,1.000000,0.164105,0.182286,14.623055,16.085797
75%,4.000000,2.500000,236.900780,987.922705,206.243032,52.025269,2.000000,0.000000,0.000000,4.000000,...,1.211149,56121.979744,10.000000,0.273454,37.000000,3.000000,0.229076,0.222141,21.908949,21.795117
max,33.000000,7.750000,932.738759,34802.675585,823.114084,447.788926,3.500000,1.000000,4.000000,5.000000,...,3.704221,870001.592216,57.750000,0.666667,115.000000,3.000000,0.914409,0.887394,68.859453,67.524964


In [11]:
import pickle

def save_df_pickle(df, filename="df_transformed.pkl"):
    with open(filename, "wb") as f:
        pickle.dump(df, f)
    print(f"✅ DataFrame sauvegardé sous {filename}")

save_df_pickle(df)


✅ DataFrame sauvegardé sous df_transformed.pkl
